In [1]:
import math

import numpy as np
import optuna
import pandas as pd
import torch
from kan import KAN
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 5
TRIALS = 3

In [3]:
data = pd.read_csv("DATA/income (IC)/Preprocessed/data_processed.csv")

y = data["target_label"].values
data.drop("target_label", axis=1, inplace=True)
X = data.values.astype(np.float32)

if y.dtype == "object":
    label_encoder = LabelEncoder()
    y = torch.tensor(label_encoder.fit_transform(y)).reshape(-1, 1).float()
else:
    y = torch.tensor(y).reshape(-1, 1).float().to(device)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, stratify=y_train, test_size=0.1, random_state=42
)


pca = PCA(n_components=0.999)
X_train = torch.tensor(pca.fit_transform(X_train)).to(device)
X_valid = torch.tensor(pca.transform(X_valid)).to(device)
X_test = torch.tensor(pca.transform(X_test)).to(device)

input_shape = X_train.shape[1]
output_shape = y_train.shape[1]

print("Reduce number of features from %d to %d." % (data.shape[1], input_shape))

dataset = {
    "train_input": X_train,
    "train_label": y_train,
    "test_input": X_valid,
    "test_label": y_valid,
}

Reduce number of features from 108 to 2.


In [5]:
def objective(trial):
    depth = trial.suggest_int("depth", 1, 3)
    grid = trial.suggest_int("grid", 1, 3)
    k = trial.suggest_int("k", 1, 4)

    width = [trial.suggest_int(f"neurons_layer_{i}", 1, 10) for i in range(depth)]
    width = [input_shape] + width + [output_shape]
    
    model = KAN(
        width=width,
        grid=grid,
        k=k,
        device=device    
    )

    # Train the model
    history = model.fit(dataset, steps=EPOCHS)
    
    # Evaluate the model
    y_score = model(X_valid).cpu()
    y_pred = (y_score > 0.5).int()
    
    return f1_score(y_valid, y_pred, average='macro')

In [6]:
# Step 3: Create a Study and Optimize
study = optuna.create_study(direction="maximize")  # Change to "minimize" if needed
study.optimize(objective, n_trials=TRIALS)  # Adjust the number of trials as needed

[I 2024-12-29 08:47:40,908] A new study created in memory with name: no-name-4ff9fecf-9986-47da-a4a4-6b004d53d6ab


checkpoint directory created: ./model
saving model version 0.0


| train_loss: 1.05e+00 | test_loss: 8.25e-01 | reg: 3.55e+01 | : 100%|█| 5/5 [01:38<00:00, 19.61s/it
[I 2024-12-29 08:49:19,104] Trial 0 finished with value: 0.38241445725119627 and parameters: {'depth': 3, 'grid': 2, 'k': 4, 'neurons_layer_0': 6, 'neurons_layer_1': 7, 'neurons_layer_2': 9}. Best is trial 0 with value: 0.38241445725119627.


saving model version 0.1
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 9.08e-01 | test_loss: 9.01e-01 | reg: 1.31e+01 | : 100%|█| 5/5 [00:24<00:00,  4.81s/it
[I 2024-12-29 08:49:43,215] Trial 1 finished with value: 0.4132991596461699 and parameters: {'depth': 1, 'grid': 1, 'k': 3, 'neurons_layer_0': 9}. Best is trial 1 with value: 0.4132991596461699.


saving model version 0.1
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 4.89e-01 | test_loss: 4.90e-01 | reg: 1.68e+01 | : 100%|█| 5/5 [00:33<00:00,  6.71s/it
[I 2024-12-29 08:50:16,842] Trial 2 finished with value: 0.5408445106609313 and parameters: {'depth': 3, 'grid': 2, 'k': 4, 'neurons_layer_0': 5, 'neurons_layer_1': 1, 'neurons_layer_2': 3}. Best is trial 2 with value: 0.5408445106609313.


saving model version 0.1


In [7]:
best_params = study.best_params
depth = best_params["depth"]
grid = best_params["grid"]
k = best_params["k"]
width = [best_params[f"neurons_layer_{i}"] for i in range(depth)]
width = [input_shape] + width + [output_shape]  # Add input and output shapes

model = KAN(
    width=width,
    grid=grid,
    k=k,
    device=device
)

# Train the model
history = model.fit(dataset, steps=EPOCHS)

y_score = model(X_test).cpu()
y_pred = (y_score > 0.5).int()

checkpoint directory created: ./model
saving model version 0.0


| train_loss: 4.89e-01 | test_loss: 4.90e-01 | reg: 1.68e+01 | : 100%|█| 5/5 [00:33<00:00,  6.78s/it

saving model version 0.1


In [8]:
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print("ROC-AUC: %.4f" % roc_auc_score(y_test, y_pred))

              precision    recall  f1-score   support

          No       0.61      0.39      0.47      4944
         Yes       0.55      0.75      0.63      4944

    accuracy                           0.57      9888
   macro avg       0.58      0.57      0.55      9888
weighted avg       0.58      0.57      0.55      9888

ROC-AUC: 0.5679
